This notebook downloads the raster data for the QGIS part of the project from Microsoft Planetary Computer's API

In [50]:
import pystac_client
import planetary_computer

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1"
)

search = catalog.search(
    collections=["sentinel-2-l2a"],
    ids=["S2B_MSIL2A_20260708T114349_R123_T29UPV_20260708T152442"]
)

item = next(search.items())

In [51]:
item.assets["B04"]

<Asset href=https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/29/U/PV/2026/07/08/S2B_MSIL2A_20260708T114349_N0512_R123_T29UPV_20260708T152442.SAFE/GRANULE/L2A_T29UPV_A048769_20260708T114435/IMG_DATA/R10m/T29UPV_20260708T114349_B04_10m.tif>

In [52]:
signed_item = planetary_computer.sign(item)

red_url = signed_item.assets["B04"].href
nir_url = signed_item.assets["B08"].href

print(red_url)
print(nir_url)

https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/29/U/PV/2026/07/08/S2B_MSIL2A_20260708T114349_N0512_R123_T29UPV_20260708T152442.SAFE/GRANULE/L2A_T29UPV_A048769_20260708T114435/IMG_DATA/R10m/T29UPV_20260708T114349_B04_10m.tif?st=2026-08-19T17%3A26%3A22Z&se=2026-08-20T18%3A11%3A22Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-08-20T16%3A29%3A24Z&ske=2026-08-27T16%3A29%3A24Z&sks=b&skv=2025-07-05&sig=DQxX/ux9Shda05uQ8XvFtV/GeNBYn5ZaAO/nAb2QUGA%3D
https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/29/U/PV/2026/07/08/S2B_MSIL2A_20260708T114349_N0512_R123_T29UPV_20260708T152442.SAFE/GRANULE/L2A_T29UPV_A048769_20260708T114435/IMG_DATA/R10m/T29UPV_20260708T114349_B08_10m.tif?st=2026-08-19T17%3A26%3A22Z&se=2026-08-20T18%3A11%3A22Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-08-20T16%3A29%3A24Z&ske=2026-08-27T16%3A29%3A24Z&sks=b&sk

Clip the files to only include a rectangle centred tight on the Phoenix Park so we don't have to download large files of data we mostly won't use.

In [53]:
import rasterio

with rasterio.open(red_url) as src:
    print("CRS:", src.crs)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Resolution:", src.res)
    print("Bounds:", src.bounds)
    print("Data type:", src.dtypes)

CRS: EPSG:32629
Width: 10980
Height: 10980
Resolution: (10.0, 10.0)
Bounds: BoundingBox(left=600000.0, bottom=5890200.0, right=709800.0, top=6000000.0)
Data type: ('uint16',)


Load the park polygon to get the dimensions of the rectangle we need

In [54]:
import geopandas as gpd

park = gpd.read_file("../data/processed/phoenix_park.gpkg")
print(park.crs)

EPSG:2157


Reproject the park polygon to the sentinel-2 crs

In [55]:
park_utm = park.to_crs("EPSG:32629")

In [56]:
print(park_utm.crs)

EPSG:32629


Download the raster data for the park (masking the pixels outside the park boundaries)

In [57]:
import rasterio
from rasterio.mask import mask
from pathlib import Path

output_dir = Path("../data/sentinel-2")
output_dir.mkdir(parents=True, exist_ok=True)

geometries = park_utm.geometry.values

with rasterio.open(red_url) as src:
    red_clip, red_transform = mask(
        src,
        geometries,
        crop=True,
        nodata=0
    )

    red_meta = src.meta.copy()

red_meta.update({
    "height": red_clip.shape[1],
    "width": red_clip.shape[2],
    "transform": red_transform,
    "nodata": 0
})

Save the clipped raster locally

In [58]:
red_path = output_dir / "S2_2026-07-08_B04_phoenix_park.tif"

with rasterio.open(red_path, "w", **red_meta) as dst:
    dst.write(red_clip)

Repeat for the NIR band

In [59]:
with rasterio.open(nir_url) as src:
    nir_clip, nir_transform = mask(
        src,
        geometries,
        crop=True,
        nodata=0
    )

    nir_meta = src.meta.copy()

nir_meta.update({
    "height": nir_clip.shape[1],
    "width": nir_clip.shape[2],
    "transform": nir_transform,
    "nodata": 0
})

nir_path = output_dir / "S2_2026-07-08_B08_phoenix_park.tif"

with rasterio.open(nir_path, "w", **nir_meta) as dst:
    dst.write(nir_clip)

In [60]:
for f in output_dir.glob("*.tif"):
    print(f.name, round(f.stat().st_size / 1024**2, 2), "MB")

S2_2026-07-08_B04_phoenix_park.tif 0.22 MB
S2_2026-07-08_B08_phoenix_park.tif 0.22 MB
S2_2026-07-08_SCL_10m_phoenix_park.tif 0.11 MB
S2_2026-07-08_SCL_phoenix_park.tif 0.03 MB
S2_2026-07-18_B04_phoenix_park.tif 0.22 MB
S2_2026-07-18_B08_phoenix_park.tif 0.22 MB


Also get the SCL raster

In [61]:
signed_item = planetary_computer.sign(item)

scl_url = signed_item.assets["SCL"].href
print(scl_url)

https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/29/U/PV/2026/07/08/S2B_MSIL2A_20260708T114349_N0512_R123_T29UPV_20260708T152442.SAFE/GRANULE/L2A_T29UPV_A048769_20260708T114435/IMG_DATA/R20m/T29UPV_20260708T114349_SCL_20m.tif?st=2026-08-19T17%3A26%3A22Z&se=2026-08-20T18%3A11%3A22Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-08-20T16%3A29%3A24Z&ske=2026-08-27T16%3A29%3A24Z&sks=b&skv=2025-07-05&sig=DQxX/ux9Shda05uQ8XvFtV/GeNBYn5ZaAO/nAb2QUGA%3D


In [62]:
with rasterio.open(scl_url) as src:
    print("CRS:", src.crs)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Resolution:", src.res)
    print("Data type:", src.dtypes)

CRS: EPSG:32629
Width: 5490
Height: 5490
Resolution: (20.0, 20.0)
Data type: ('uint8',)


In [63]:
with rasterio.open(scl_url) as src:
    scl_clip, scl_transform = mask(
        src,
        geometries,
        crop=True,
        nodata=0
    )

    scl_meta = src.meta.copy()

scl_meta.update({
    "height": scl_clip.shape[1],
    "width": scl_clip.shape[2],
    "transform": scl_transform,
    "nodata": 0
})

scl_path = output_dir / "S2_2026-07-08_SCL_phoenix_park.tif"

with rasterio.open(scl_path, "w", **scl_meta) as dst:
    dst.write(scl_clip)

Let's also grab the raster data for 18th July (the latest date in July 2026 without cloud cover) to see how the NDVI changes over these 10 hot days in July

S2B_MSIL2A_20260718T114349_R123_T29UPV_20260718T152635

In [64]:
search = catalog.search(
    collections=["sentinel-2-l2a"],
    ids=["S2B_MSIL2A_20260718T114349_R123_T29UPV_20260718T152635"]
)

item = next(search.items())

In [65]:
signed_item = planetary_computer.sign(item)

red_url = signed_item.assets["B04"].href
nir_url = signed_item.assets["B08"].href
scl_url = signed_item.assets["SCL"].href

print(red_url)
print(nir_url)
print(scl_url)

https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/29/U/PV/2026/07/18/S2B_MSIL2A_20260718T114349_N0512_R123_T29UPV_20260718T152635.SAFE/GRANULE/L2A_T29UPV_A048912_20260718T114347/IMG_DATA/R10m/T29UPV_20260718T114349_B04_10m.tif?st=2026-08-19T17%3A26%3A22Z&se=2026-08-20T18%3A11%3A22Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-08-20T16%3A29%3A24Z&ske=2026-08-27T16%3A29%3A24Z&sks=b&skv=2025-07-05&sig=DQxX/ux9Shda05uQ8XvFtV/GeNBYn5ZaAO/nAb2QUGA%3D
https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/29/U/PV/2026/07/18/S2B_MSIL2A_20260718T114349_N0512_R123_T29UPV_20260718T152635.SAFE/GRANULE/L2A_T29UPV_A048912_20260718T114347/IMG_DATA/R10m/T29UPV_20260718T114349_B08_10m.tif?st=2026-08-19T17%3A26%3A22Z&se=2026-08-20T18%3A11%3A22Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-08-20T16%3A29%3A24Z&ske=2026-08-27T16%3A29%3A24Z&sks=b&sk

Clip and save red band for 18th July

In [66]:
with rasterio.open(red_url) as src:
    red_clip, red_transform = mask(
        src,
        geometries,
        crop=True,
        nodata=0
    )

    red_meta = src.meta.copy()

red_meta.update({
    "height": red_clip.shape[1],
    "width": red_clip.shape[2],
    "transform": red_transform,
    "nodata": 0
})

red_path = output_dir / "S2_2026-07-18_B04_phoenix_park.tif"

with rasterio.open(red_path, "w", **red_meta) as dst:
    dst.write(red_clip)

NIR band

In [67]:
with rasterio.open(nir_url) as src:
    nir_clip, nir_transform = mask(
        src,
        geometries,
        crop=True,
        nodata=0
    )

    nir_meta = src.meta.copy()

nir_meta.update({
    "height": nir_clip.shape[1],
    "width": nir_clip.shape[2],
    "transform": nir_transform,
    "nodata": 0
})

nir_path = output_dir / "S2_2026-07-18_B08_phoenix_park.tif"

with rasterio.open(nir_path, "w", **nir_meta) as dst:
    dst.write(nir_clip)

SCL band

In [68]:
with rasterio.open(scl_url) as src:
    scl_clip, scl_transform = mask(
        src,
        geometries,
        crop=True,
        nodata=0
    )

    scl_meta = src.meta.copy()

scl_meta.update({
    "height": scl_clip.shape[1],
    "width": scl_clip.shape[2],
    "transform": scl_transform,
    "nodata": 0
})

scl_path = output_dir / "S2_2026-07-18_SCL_phoenix_park.tif"

with rasterio.open(scl_path, "w", **scl_meta) as dst:
    dst.write(scl_clip)